# Amazon Employee Access: Optuna training




In [1]:
from pathlib import Path
import time

import pandas as pd
import numpy as np
import joblib
import optuna

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss
)

PROJECT_DIR = Path("C:/Case Study Trees/AmazonEmployeeStudy")

SPLIT_DIR = PROJECT_DIR / "data" / "splits"
PREPROCESSOR_DIR = PROJECT_DIR / "artifacts" / "preprocessors"
MODEL_DIR = PROJECT_DIR / "artifacts" / "models"
INPUT_INFO_DIR = PROJECT_DIR / "artifacts" / "input_info"
OPTUNA_DIR = PROJECT_DIR / "optuna"
VALIDATION_DIR = PROJECT_DIR / "results" / "validation"

RANDOM_STATE = 42
N_THREADS = 4
TUNING_SECONDS = 10 * 60

c:\Users\gdeep\anaconda3\envs\100DaysML_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data = pd.read_csv(SPLIT_DIR / "train.csv")
valid_data = pd.read_csv(SPLIT_DIR / "valid.csv")

dropped_columns = joblib.load(
    PREPROCESSOR_DIR / "dropped_unique_columns.joblib"
)

feature_columns = [
    column for column in train_data.columns
    if column not in ["ACTION", "ROW_ID"] + dropped_columns
]

X_train = pd.read_csv(
    SPLIT_DIR / "train.csv",
    dtype={column: str for column in feature_columns}
)[feature_columns]

X_valid = pd.read_csv(
    SPLIT_DIR / "valid.csv",
    dtype={column: str for column in feature_columns}
)[feature_columns]

y_train = train_data["ACTION"]
y_valid = valid_data["ACTION"]

categorical_columns = feature_columns.copy()

onehot_preprocessor = joblib.load(
    PREPROCESSOR_DIR / "rf_onehot_preprocessor.joblib"
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)

print("\nCategorical features:")
print(categorical_columns)

print("\nOptuna budget per model:", TUNING_SECONDS, "seconds")
print("CPU threads per model:", N_THREADS)

Training shape: (22938, 9)
Validation shape: (4915, 9)

Categorical features:
['RESOURCE', 'MGR_ID', 'ROLE_ROLLUP_1', 'ROLE_ROLLUP_2', 'ROLE_DEPTNAME', 'ROLE_TITLE', 'ROLE_FAMILY_DESC', 'ROLE_FAMILY', 'ROLE_CODE']

Optuna budget per model: 600 seconds
CPU threads per model: 4


## RANDOM FOREST

In [3]:
X_train_rf = onehot_preprocessor.transform(X_train)
X_valid_rf = onehot_preprocessor.transform(X_valid)

print("Encoded training shape:", X_train_rf.shape)
print("Encoded validation shape:", X_valid_rf.shape)


def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 400, step=100
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 10, 40, step=5
        ),
        "min_samples_split": trial.suggest_int(
            "min_samples_split", 2, 20
        ),
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf", 1, 10
        ),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", 0.1, 0.3]
        ),
        "class_weight": trial.suggest_categorical(
            "class_weight", [None, "balanced"]
        )
    }

    model = RandomForestClassifier(
        **params,
        random_state=RANDOM_STATE,
        n_jobs=N_THREADS
    )

    start = time.perf_counter()
    model.fit(X_train_rf, y_train)
    fit_seconds = time.perf_counter() - start

    probabilities = model.predict_proba(X_valid_rf)[:, 1]

    trial.set_user_attr("Fit_Seconds", fit_seconds)
    trial.set_user_attr(
        "Average_Precision",
        float(average_precision_score(y_valid, probabilities))
    )
    trial.set_user_attr(
        "Log_Loss",
        float(log_loss(y_valid, probabilities))
    )

    return float(log_loss(y_valid, probabilities))

Encoded training shape: (22938, 13694)
Encoded validation shape: (4915, 13694)


In [4]:
rf_study = optuna.create_study(
    study_name="amazon_rf_onehot_logloss_v1",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    storage=f"sqlite:///{(OPTUNA_DIR / 'amazon_rf.db').as_posix()}",
    load_if_exists=True
)

start = time.perf_counter()

rf_study.optimize(
    rf_objective,
    timeout=TUNING_SECONDS,
    n_jobs=1,
    gc_after_trial=True
)

rf_tuning_seconds = time.perf_counter() - start

rf_study.trials_dataframe().to_csv(
    VALIDATION_DIR / "rf_optuna_trials.csv",
    index=False
)

print("Best trial:", rf_study.best_trial.number)
print("Best validation log loss:", round(rf_study.best_value, 6))
print("Tuning seconds:", round(rf_tuning_seconds, 2))

print("\nBest parameters:")
for name, value in rf_study.best_params.items():
    print(f"{name}: {value}")

[I 2026-10-16 20:30:47,927] A new study created in RDB with name: amazon_rf_onehot_logloss_v1
[I 2026-10-16 20:30:49,365] Trial 0 finished with value: 0.1933602600904449 and parameters: {'n_estimators': 200, 'max_depth': 40, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 0 with value: 0.1933602600904449.
[I 2026-10-16 20:30:51,216] Trial 1 finished with value: 0.6609583776027025 and parameters: {'n_estimators': 300, 'max_depth': 10, 'min_samples_split': 20, 'min_samples_leaf': 9, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.1933602600904449.
[I 2026-10-16 20:30:56,110] Trial 2 finished with value: 0.19789671629674876 and parameters: {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 13, 'min_samples_leaf': 2, 'max_features': 0.3, 'class_weight': None}. Best is trial 0 with value: 0.1933602600904449.
[I 2026-10-16 20:31:06,128] Trial 3 finished with value: 0.1872034982011709 and 

Best trial: 15
Best validation log loss: 0.181329
Tuning seconds: 602.57

Best parameters:
n_estimators: 100
max_depth: 40
min_samples_split: 9
min_samples_leaf: 6
max_features: 0.3
class_weight: None


In [5]:
rf_best = RandomForestClassifier(
    **rf_study.best_params,
    random_state=RANDOM_STATE,
    n_jobs=N_THREADS
)

start = time.perf_counter()
rf_best.fit(X_train_rf, y_train)
rf_fit_seconds = time.perf_counter() - start

rf_valid_probability = rf_best.predict_proba(X_valid_rf)[:, 1]

rf_results = {
    "Model": "Random Forest",
    "Representation": "One-hot",
    "Validation_Log_Loss": log_loss(
        y_valid, rf_valid_probability
    ),
    "Validation_ROC_AUC": roc_auc_score(
        y_valid, rf_valid_probability
    ),
    "Validation_AP_Approval": average_precision_score(
        y_valid, rf_valid_probability
    ),
    "Validation_AP_Denial": average_precision_score(
        1 - y_valid, 1 - rf_valid_probability
    ),
    "Fit_Seconds": rf_fit_seconds,
    "Tuning_Seconds": rf_tuning_seconds,
    "Completed_Trials": sum(
        trial.state == optuna.trial.TrialState.COMPLETE
        for trial in rf_study.trials
    ),
    "CPU_Threads": N_THREADS
}

joblib.dump(
    rf_best,
    MODEL_DIR / "amazon_rf_tuned.joblib"
)

joblib.dump(
    {
        "feature_columns": feature_columns,
        "categorical_columns": categorical_columns,
        "dropped_columns": dropped_columns,
        "preprocessor_path": "artifacts/preprocessors/rf_onehot_preprocessor.joblib",
        "positive_class": 1,
        "best_params": rf_study.best_params
    },
    INPUT_INFO_DIR / "amazon_rf_input_info.joblib"
)

pd.DataFrame([rf_results]).to_csv(
    VALIDATION_DIR / "rf_tuned_validation.csv",
    index=False
)

display(pd.DataFrame([rf_results]).round(4))
print("Best Random Forest and validation results saved.")

,Model,Representation,Validation_Log_Loss,Validation_ROC_AUC,Validation_AP_Approval,Validation_AP_Denial,Fit_Seconds,Tuning_Seconds,Completed_Trials,CPU_Threads
0,Random Forest,One-hot,0.1813,0.8001,0.9768,0.3116,4.4856,602.5736,75,4


Best Random Forest and validation results saved.


## XGBOOST

In [6]:
from xgboost import XGBClassifier
import xgboost

print("XGBoost version:", xgboost.__version__)

X_train_xgb = X_train.copy()
X_valid_xgb = X_valid.copy()

xgb_category_levels = {}

for column in categorical_columns:
    levels = X_train[column].unique().tolist()
    xgb_category_levels[column] = levels

    category_type = pd.CategoricalDtype(categories=levels)

    X_train_xgb[column] = X_train[column].astype(category_type)
    X_valid_xgb[column] = X_valid[column].astype(category_type)

print("\nCategorical features:")
print(categorical_columns)

print("\nValidation values unseen during training:")
print(X_valid_xgb.isna().sum())

joblib.dump(
    xgb_category_levels,
    INPUT_INFO_DIR / "amazon_xgb_category_levels.joblib"
)

XGBoost version: 3.2.0

Categorical features:
['RESOURCE', 'MGR_ID', 'ROLE_ROLLUP_1', 'ROLE_ROLLUP_2', 'ROLE_DEPTNAME', 'ROLE_TITLE', 'ROLE_FAMILY_DESC', 'ROLE_FAMILY', 'ROLE_CODE']

Validation values unseen during training:
RESOURCE            713
MGR_ID              212
ROLE_ROLLUP_1         3
ROLE_ROLLUP_2         6
ROLE_DEPTNAME         8
ROLE_TITLE           14
ROLE_FAMILY_DESC    160
ROLE_FAMILY           2
ROLE_CODE            14
dtype: int64


['C:\\Case Study Trees\\AmazonEmployeeStudy\\artifacts\\input_info\\amazon_xgb_category_levels.joblib']

In [9]:
def xgb_objective(trial):
    params = {
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.02, 0.2, log=True
        ),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1, 30, log=True
        ),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 10, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 0.1, 30, log=True
        )
    }

    model = XGBClassifier(
        **params,
        n_estimators=2000,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        enable_categorical=True,
        early_stopping_rounds=50,
        random_state=RANDOM_STATE,
        n_jobs=N_THREADS
    )

    start = time.perf_counter()

    model.fit(
        X_train_xgb,
        y_train,
        eval_set=[(X_valid_xgb, y_valid)],
        verbose=False
    )

    fit_seconds = time.perf_counter() - start
    probabilities = model.predict_proba(X_valid_xgb)[:, 1]

    trial.set_user_attr("Best_Trees", int(model.best_iteration + 1))
    trial.set_user_attr("Fit_Seconds", fit_seconds)
    trial.set_user_attr(
        "ROC_AUC",
        float(roc_auc_score(y_valid, probabilities))
    )
    trial.set_user_attr(
        "Average_Precision",
        float(average_precision_score(y_valid, probabilities))
    )

    return float(log_loss(y_valid, probabilities))

In [10]:
xgb_study = optuna.create_study(
    study_name="amazon_xgb_native_logloss_v1",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    storage=f"sqlite:///{(OPTUNA_DIR / 'amazon_xgb.db').as_posix()}",
    load_if_exists=True
)

start = time.perf_counter()

xgb_study.optimize(
    xgb_objective,
    timeout=TUNING_SECONDS,
    n_jobs=1,
    gc_after_trial=True
)

xgb_tuning_seconds = time.perf_counter() - start

xgb_study.trials_dataframe().to_csv(
    VALIDATION_DIR / "xgb_optuna_trials.csv",
    index=False
)

print("Best trial:", xgb_study.best_trial.number)
print("Best validation log loss:", round(xgb_study.best_value, 6))
print("Best trees:", xgb_study.best_trial.user_attrs["Best_Trees"])
print("Tuning seconds:", round(xgb_tuning_seconds, 2))

print("\nBest parameters:")
for name, value in xgb_study.best_params.items():
    print(f"{name}: {value}")

[I 2026-10-16 20:45:26,090] Using an existing study with `study_name='amazon_xgb_native_logloss_v1'` instead of creating a new one.
[I 2026-10-16 20:45:41,428] Trial 0 finished with value: 0.1730165034532547 and parameters: {'learning_rate': 0.047377279007281566, 'max_depth': 10, 'min_child_weight': 12.057126287443763, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'reg_alpha': 0.000602521573620386, 'reg_lambda': 0.13927723945289014}. Best is trial 0 with value: 0.1730165034532547.
[I 2026-10-16 20:45:48,044] Trial 1 finished with value: 0.17618773877620697 and parameters: {'learning_rate': 0.146962368105409, 'max_depth': 7, 'min_child_weight': 11.114989443094977, 'subsample': 0.7061753482887407, 'colsample_bytree': 0.9909729556485982, 'reg_alpha': 1.452824663751602, 'reg_lambda': 0.33572967053517927}. Best is trial 0 with value: 0.1730165034532547.
[I 2026-10-16 20:45:56,892] Trial 2 finished with value: 0.16417713463306427 and parameters: {'learning_rate': 0

Best trial: 37
Best validation log loss: 0.154556
Best trees: 229
Tuning seconds: 602.25

Best parameters:
learning_rate: 0.02357857665321209
max_depth: 9
min_child_weight: 1.0100861618286068
subsample: 0.9235613048702429
colsample_bytree: 0.807982076428852
reg_alpha: 0.058952969338010315
reg_lambda: 1.3222429354377578


In [11]:
xgb_best_trees = xgb_study.best_trial.user_attrs["Best_Trees"]

xgb_best = XGBClassifier(
    **xgb_study.best_params,
    n_estimators=xgb_best_trees,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    enable_categorical=True,
    random_state=RANDOM_STATE,
    n_jobs=N_THREADS
)
start = time.perf_counter()
xgb_best.fit(X_train_xgb, y_train)
xgb_fit_seconds = time.perf_counter() - start

xgb_valid_probability = xgb_best.predict_proba(X_valid_xgb)[:, 1]

xgb_results = {
    "Model": "XGBoost",
    "Representation": "Native",
    "Validation_Log_Loss": log_loss(
        y_valid, xgb_valid_probability
    ),
    "Validation_ROC_AUC": roc_auc_score(
        y_valid, xgb_valid_probability
    ),
    "Validation_AP_Approval": average_precision_score(
        y_valid, xgb_valid_probability
    ),
    "Validation_AP_Denial": average_precision_score(
        1 - y_valid, 1 - xgb_valid_probability
    ),
    "Fit_Seconds": xgb_fit_seconds,
    "Tuning_Seconds": xgb_tuning_seconds,
    "Completed_Trials": sum(
        trial.state == optuna.trial.TrialState.COMPLETE
        for trial in xgb_study.trials
    ),
    "CPU_Threads": N_THREADS
}

joblib.dump(
    xgb_best,
    MODEL_DIR / "amazon_xgb_tuned.joblib"
)
xgb_best.save_model(
    str(MODEL_DIR / "amazon_xgb_tuned.json")
)

joblib.dump(
    {
        "feature_columns": feature_columns,
        "categorical_columns": categorical_columns,
        "category_levels": xgb_category_levels,
        "dropped_columns": dropped_columns,
        "positive_class": 1,
        "best_params": xgb_study.best_params,
        "best_trees": xgb_best_trees
    },
    INPUT_INFO_DIR / "amazon_xgb_input_info.joblib"
)

pd.DataFrame([xgb_results]).to_csv(
    VALIDATION_DIR / "xgb_tuned_validation.csv",
    index=False
)

display(pd.DataFrame([xgb_results]).round(4))
print("Best XGBoost model and validation results saved.")

,Model,Representation,Validation_Log_Loss,Validation_ROC_AUC,Validation_AP_Approval,Validation_AP_Denial,Fit_Seconds,Tuning_Seconds,Completed_Trials,CPU_Threads
0,XGBoost,Native,0.1546,0.8591,0.9858,0.4365,13.8382,602.2497,40,4


Best XGBoost model and validation results saved.


## LIGHTGBM

In [12]:
import lightgbm as lgb
from lightgbm import LGBMClassifier

print("LightGBM version:", lgb.__version__)

X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()

lgb_category_levels = {}

for column in categorical_columns:
    levels = xgb_category_levels[column]
    lgb_category_levels[column] = levels

    category_type = pd.CategoricalDtype(categories=levels)

    X_train_lgb[column] = X_train[column].astype(category_type)
    X_valid_lgb[column] = X_valid[column].astype(category_type)

print("\nCategorical features:")
print(categorical_columns)

print("\nValidation values unseen during training:")
print(X_valid_lgb.isna().sum())

joblib.dump(
    lgb_category_levels,
    INPUT_INFO_DIR / "amazon_lgb_category_levels.joblib"
)

LightGBM version: 4.7.0

Categorical features:
['RESOURCE', 'MGR_ID', 'ROLE_ROLLUP_1', 'ROLE_ROLLUP_2', 'ROLE_DEPTNAME', 'ROLE_TITLE', 'ROLE_FAMILY_DESC', 'ROLE_FAMILY', 'ROLE_CODE']

Validation values unseen during training:
RESOURCE            713
MGR_ID              212
ROLE_ROLLUP_1         3
ROLE_ROLLUP_2         6
ROLE_DEPTNAME         8
ROLE_TITLE           14
ROLE_FAMILY_DESC    160
ROLE_FAMILY           2
ROLE_CODE            14
dtype: int64


['C:\\Case Study Trees\\AmazonEmployeeStudy\\artifacts\\input_info\\amazon_lgb_category_levels.joblib']

In [13]:
def lgb_objective(trial):
    max_depth = trial.suggest_int("max_depth", 3, 10)

    params = {
        "max_depth": max_depth,
        "num_leaves": trial.suggest_int(
            "num_leaves", 8, min(128, 2 ** max_depth)
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.02, 0.2, log=True
        ),
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 10, 100
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.7, 1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 10, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 0.1, 30, log=True
        ),
        "cat_smooth": trial.suggest_float(
            "cat_smooth", 1, 50, log=True
        )
    }

    model = LGBMClassifier(
        **params,
        n_estimators=2000,
        objective="binary",
        subsample_freq=1,
        random_state=RANDOM_STATE,
        n_jobs=N_THREADS,
        verbosity=-1
    )

    start = time.perf_counter()

    model.fit(
        X_train_lgb,
        y_train,
        categorical_feature=categorical_columns,
        eval_set=[(X_valid_lgb, y_valid)],
        eval_metric="binary_logloss",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=50,
                first_metric_only=True,
                verbose=False
            )
        ]
    )

    fit_seconds = time.perf_counter() - start
    probabilities = model.predict_proba(X_valid_lgb)[:, 1]

    trial.set_user_attr("Best_Trees", int(model.best_iteration_))
    trial.set_user_attr("Fit_Seconds", fit_seconds)
    trial.set_user_attr(
        "ROC_AUC",
        float(roc_auc_score(y_valid, probabilities))
    )
    trial.set_user_attr(
        "Average_Precision",
        float(average_precision_score(y_valid, probabilities))
    )

    return float(log_loss(y_valid, probabilities))

In [14]:
lgb_study = optuna.create_study(
    study_name="amazon_lgb_native_logloss_v1",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    storage=f"sqlite:///{(OPTUNA_DIR / 'amazon_lgb.db').as_posix()}",
    load_if_exists=True
)

start = time.perf_counter()

lgb_study.optimize(
    lgb_objective,
    timeout=TUNING_SECONDS,
    n_jobs=1,
    gc_after_trial=True
)

lgb_tuning_seconds = time.perf_counter() - start

lgb_study.trials_dataframe().to_csv(
    VALIDATION_DIR / "lgb_optuna_trials.csv",
    index=False
)

print("Best trial:", lgb_study.best_trial.number)
print("Best validation log loss:", round(lgb_study.best_value, 6))
print("Best trees:", lgb_study.best_trial.user_attrs["Best_Trees"])
print("Tuning seconds:", round(lgb_tuning_seconds, 2))

print("\nBest parameters:")
for name, value in lgb_study.best_params.items():
    print(f"{name}: {value}")

[I 2026-10-16 20:57:28,049] A new study created in RDB with name: amazon_lgb_native_logloss_v1
c:\Users\gdeep\anaconda3\envs\100DaysML_311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
[I 2026-10-16 20:57:29,010] Trial 0 finished with value: 0.15834121024995557 and parameters: {'max_depth': 5, 'num_leaves': 31, 'learning_rate': 0.10790061933340454, 'min_child_samples': 64, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 13.983740016490973, 'cat_smooth': 10.502105436744278}. Best is trial 0 with value: 0.15834121024995557.
c:\Users\gdeep\anaconda3\envs\100DaysML_311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_se

Best trial: 171
Best validation log loss: 0.15181
Best trees: 273
Tuning seconds: 600.02

Best parameters:
max_depth: 10
num_leaves: 123
learning_rate: 0.0210389473814344
min_child_samples: 22
subsample: 0.7497403724073001
colsample_bytree: 0.7481699927303478
reg_alpha: 0.0001870127785258041
reg_lambda: 29.556698914518176
cat_smooth: 1.3551257492181579


In [15]:
lgb_best_trees = lgb_study.best_trial.user_attrs["Best_Trees"]

lgb_best = LGBMClassifier(
    **lgb_study.best_params,
    n_estimators=lgb_best_trees,
    objective="binary",
    subsample_freq=1,
    random_state=RANDOM_STATE,
    n_jobs=N_THREADS,
    verbosity=-1
)

start = time.perf_counter()

lgb_best.fit(
    X_train_lgb,
    y_train,
    categorical_feature=categorical_columns
)

lgb_fit_seconds = time.perf_counter() - start

lgb_valid_probability = lgb_best.predict_proba(X_valid_lgb)[:, 1]

lgb_results = {
    "Model": "LightGBM",
    "Representation": "Native",
    "Validation_Log_Loss": log_loss(
        y_valid, lgb_valid_probability
    ),
    "Validation_ROC_AUC": roc_auc_score(
        y_valid, lgb_valid_probability
    ),
    "Validation_AP_Approval": average_precision_score(
        y_valid, lgb_valid_probability
    ),
    "Validation_AP_Denial": average_precision_score(
        1 - y_valid, 1 - lgb_valid_probability
    ),
    "Fit_Seconds": lgb_fit_seconds,
    "Tuning_Seconds": lgb_tuning_seconds,
    "Completed_Trials": sum(
        trial.state == optuna.trial.TrialState.COMPLETE
        for trial in lgb_study.trials
    ),
    "CPU_Threads": N_THREADS
}

joblib.dump(
    lgb_best,
    MODEL_DIR / "amazon_lgb_tuned.joblib"
)

joblib.dump(
    {
        "feature_columns": feature_columns,
        "categorical_columns": categorical_columns,
        "category_levels": lgb_category_levels,
        "dropped_columns": dropped_columns,
        "positive_class": 1,
        "best_params": lgb_study.best_params,
        "best_trees": lgb_best_trees
    },
    INPUT_INFO_DIR / "amazon_lgb_input_info.joblib"
)

pd.DataFrame([lgb_results]).to_csv(
    VALIDATION_DIR / "lgb_tuned_validation.csv",
    index=False
)

display(pd.DataFrame([lgb_results]).round(4))
print("Best LightGBM model and validation results saved.")

,Model,Representation,Validation_Log_Loss,Validation_ROC_AUC,Validation_AP_Approval,Validation_AP_Denial,Fit_Seconds,Tuning_Seconds,Completed_Trials,CPU_Threads
0,LightGBM,Native,0.1518,0.8625,0.9853,0.4493,1.6792,600.0172,277,4


Best LightGBM model and validation results saved.


## CATBOOST

In [16]:
import catboost
from catboost import CatBoostClassifier

print("CatBoost version:", catboost.__version__)

X_train_cat = X_train.copy()
X_valid_cat = X_valid.copy()


for column in categorical_columns:
    X_train_cat[column] = (
        X_train_cat[column].fillna("Missing").astype(str)
    )
    X_valid_cat[column] = (
        X_valid_cat[column].fillna("Missing").astype(str)
    )

print("\nCategorical features:")
print(categorical_columns)

print("\nTraining shape:", X_train_cat.shape)
print("Validation shape:", X_valid_cat.shape)

display(X_train_cat.head())

CatBoost version: 1.2.10

Categorical features:
['RESOURCE', 'MGR_ID', 'ROLE_ROLLUP_1', 'ROLE_ROLLUP_2', 'ROLE_DEPTNAME', 'ROLE_TITLE', 'ROLE_FAMILY_DESC', 'ROLE_FAMILY', 'ROLE_CODE']

Training shape: (22938, 9)
Validation shape: (4915, 9)


,RESOURCE,MGR_ID,ROLE_ROLLUP_1,ROLE_ROLLUP_2,ROLE_DEPTNAME,ROLE_TITLE,ROLE_FAMILY_DESC,ROLE_FAMILY,ROLE_CODE
0,73110,4945,117961,118300,118360,118129,122064,118131,118132
1,13878,13037,118006,118007,118008,117879,117897,19721,117880
2,40855,7560,117961,118300,119195,117905,117906,290919,117908
3,41289,71401,91261,118026,118202,118784,124402,290919,118786
4,17183,4828,117961,118343,118395,118890,125128,118398,118892


In [17]:
def cat_objective(trial):
    params = {
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.02, 0.2, log=True
        ),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1, 30, log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0.01, 10, log=True
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0, 5
        ),
        "max_ctr_complexity": trial.suggest_int(
            "max_ctr_complexity", 1, 2
        )
    }

    model = CatBoostClassifier(
        **params,
        iterations=2000,
        loss_function="Logloss",
        eval_metric="Logloss",
        bootstrap_type="Bayesian",
        cat_features=categorical_columns,
        random_seed=RANDOM_STATE,
        thread_count=N_THREADS,
        allow_writing_files=False,
        verbose=False
    )

    start = time.perf_counter()

    model.fit(
        X_train_cat,
        y_train,
        eval_set=(X_valid_cat, y_valid),
        early_stopping_rounds=50,
        use_best_model=True,
        verbose=False
    )

    fit_seconds = time.perf_counter() - start
    probabilities = model.predict_proba(X_valid_cat)[:, 1]

    trial.set_user_attr("Best_Trees", int(model.tree_count_))
    trial.set_user_attr("Fit_Seconds", fit_seconds)
    trial.set_user_attr(
        "ROC_AUC",
        float(roc_auc_score(y_valid, probabilities))
    )
    trial.set_user_attr(
        "Average_Precision",
        float(average_precision_score(y_valid, probabilities))
    )

    return float(log_loss(y_valid, probabilities))

In [18]:
cat_study = optuna.create_study(
    study_name="amazon_cat_native_logloss_v1",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    storage=f"sqlite:///{(OPTUNA_DIR / 'amazon_cat.db').as_posix()}",
    load_if_exists=True
)

start = time.perf_counter()

cat_study.optimize(
    cat_objective,
    timeout=TUNING_SECONDS,
    n_jobs=1,
    gc_after_trial=True
)

cat_tuning_seconds = time.perf_counter() - start

cat_study.trials_dataframe().to_csv(
    VALIDATION_DIR / "cat_optuna_trials.csv",
    index=False
)

print("Best trial:", cat_study.best_trial.number)
print("Best validation log loss:", round(cat_study.best_value, 6))
print("Best trees:", cat_study.best_trial.user_attrs["Best_Trees"])
print("Tuning seconds:", round(cat_tuning_seconds, 2))

print("\nBest parameters:")
for name, value in cat_study.best_params.items():
    print(f"{name}: {value}")

[I 2026-10-16 21:15:18,010] A new study created in RDB with name: amazon_cat_native_logloss_v1
[I 2026-10-16 21:15:57,206] Trial 0 finished with value: 0.156416313821345 and parameters: {'learning_rate': 0.047377279007281566, 'depth': 8, 'l2_leaf_reg': 12.057126287443763, 'random_strength': 0.6251373574521749, 'bagging_temperature': 0.7800932022121826, 'max_ctr_complexity': 1}. Best is trial 0 with value: 0.156416313821345.
[I 2026-10-16 21:17:25,581] Trial 1 finished with value: 0.1397948452616809 and parameters: {'learning_rate': 0.022861967752626437, 'depth': 8, 'l2_leaf_reg': 7.725378389307352, 'random_strength': 1.3311216080736887, 'bagging_temperature': 0.10292247147901223, 'max_ctr_complexity': 2}. Best is trial 1 with value: 0.1397948452616809.
[I 2026-10-16 21:17:50,212] Trial 2 finished with value: 0.14304531666894887 and parameters: {'learning_rate': 0.1359792484318226, 'depth': 5, 'l2_leaf_reg': 1.8559980846490576, 'random_strength': 0.03549878832196503, 'bagging_temperatur

Best trial: 1
Best validation log loss: 0.139795
Best trees: 832
Tuning seconds: 629.9

Best parameters:
learning_rate: 0.022861967752626437
depth: 8
l2_leaf_reg: 7.725378389307352
random_strength: 1.3311216080736887
bagging_temperature: 0.10292247147901223
max_ctr_complexity: 2


In [19]:
cat_best_trees = cat_study.best_trial.user_attrs["Best_Trees"]

cat_best = CatBoostClassifier(
    **cat_study.best_params,
    iterations=cat_best_trees,
    loss_function="Logloss",
    eval_metric="Logloss",
    bootstrap_type="Bayesian",
    cat_features=categorical_columns,
    random_seed=RANDOM_STATE,
    thread_count=N_THREADS,
    allow_writing_files=False,
    verbose=False
)

start = time.perf_counter()
cat_best.fit(X_train_cat, y_train)
cat_fit_seconds = time.perf_counter() - start

cat_valid_probability = cat_best.predict_proba(X_valid_cat)[:, 1]

cat_results = {
    "Model": "CatBoost",
    "Representation": "Native",
    "Validation_Log_Loss": log_loss(
        y_valid, cat_valid_probability
    ),
    "Validation_ROC_AUC": roc_auc_score(
        y_valid, cat_valid_probability
    ),
    "Validation_AP_Approval": average_precision_score(
        y_valid, cat_valid_probability
    ),
    "Validation_AP_Denial": average_precision_score(
        1 - y_valid, 1 - cat_valid_probability
    ),
    "Fit_Seconds": cat_fit_seconds,
    "Tuning_Seconds": cat_tuning_seconds,
    "Completed_Trials": sum(
        trial.state == optuna.trial.TrialState.COMPLETE
        for trial in cat_study.trials
    ),
    "CPU_Threads": N_THREADS
}

joblib.dump(
    cat_best,
    MODEL_DIR / "amazon_cat_tuned.joblib"
)

cat_best.save_model(
    str(MODEL_DIR / "amazon_cat_tuned.cbm")
)

joblib.dump(
    {
        "feature_columns": feature_columns,
        "categorical_columns": categorical_columns,
        "dropped_columns": dropped_columns,
        "missing_fill_value": "Missing",
        "positive_class": 1,
        "best_params": cat_study.best_params,
        "best_trees": cat_best_trees
    },
    INPUT_INFO_DIR / "amazon_cat_input_info.joblib"
)

pd.DataFrame([cat_results]).to_csv(
    VALIDATION_DIR / "cat_tuned_validation.csv",
    index=False
)

display(pd.DataFrame([cat_results]).round(4))
print("Best CatBoost model and validation results saved.")

,Model,Representation,Validation_Log_Loss,Validation_ROC_AUC,Validation_AP_Approval,Validation_AP_Denial,Fit_Seconds,Tuning_Seconds,Completed_Trials,CPU_Threads
0,CatBoost,Native,0.1398,0.8962,0.9908,0.5093,89.9499,629.8959,13,4


Best CatBoost model and validation results saved.
